<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 01 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">Daily Order Summary</div>
  <p class="doris-cover-lead">Start with six orders, filter invalid statuses, and build daily sales and a monthly asynchronous materialized view.</p>
  <span class="doris-cover-note">Source · Table · Data Test · Partition · Bucket · Async MV</span>
</div>

## 1. Check the execution environment

Run this cell first. It uses the Demo dbt environment and current Doris connection settings, then confirms that a Backend is available.

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("Start Jupyter from the dbt-for-apache-doris repository or a subdirectory.")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 1: Daily Order Summary

Sales operations needs daily valid-order volume and revenue, while finance needs the same definitions rolled up by month. This Demo excludes cancelled, returned, and failed orders and delivers daily sales plus a monthly summary view.

<table class="doris-index">
  <tr><th>Business users</th><td>Sales operations and finance</td></tr>
  <tr><th>Business question</th><td>How many valid orders and how much valid-order revenue did we record each day and month?</td></tr>
  <tr><th>Metric rule</th><td>Group by order date; exclude CANCELLED, RETURNED, and FAILED</td></tr>
  <tr><th>Delivered datasets</th><td>Daily sales table <code>daily_order_summary</code>; monthly summary <code>monthly_order_summary_mv</code></td></tr>
</table>

The six cells below show the source orders, business filter, daily aggregation, data quality checks, and monthly summary in order.

<div class="doris-flow">
  <div class="doris-flow-step"><strong>Source orders</strong>6 Doris orders</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Filter</strong>Exclude cancelled, returned, and failed orders</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Daily summary</strong><code>daily_order_summary</code> · 3 rows</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Monthly view</strong><code>monthly_order_summary_mv</code> · 1 row</div>
</div>

### 2.1 Prepare and inspect source orders

The fixture creates six orders. The final column shows whether each record enters the model or is filtered out.

In [ ]:
daily_demo_dir = runner.examples_root / "doris-daily-order-summary"
runner.show_file("Fixture SQL", daily_demo_dir / "scripts/setup.sql")
runner.run_sql_file("Create source orders", daily_demo_dir / "scripts/setup.sql")
runner.query("Input: 6 raw orders", """
select
    order_id, ordered_at, grand_total, status,
    case
        when status in ('CANCELLED', 'RETURNED', 'FAILED') then 'Filtered'
        else 'Included in daily summary'
    end as transform_action
from dbt_demo_daily_source.orders
order by order_id
""")

### 2.2 Declare the Doris table as a dbt Source

`sources.yml` maps `source('orders', 'orders')` to Doris table `dbt_demo_daily_source.orders`. `dbt ls` resolves the Source node and confirms that the project recognizes the source table. The connection was checked in the environment step.

In [ ]:
runner.show_file("dbt Source configuration", daily_demo_dir / "models/staging/sources.yml")
runner.run_dbt("Resolve the Demo Source node", daily_demo_dir, "ls", "--resource-type", "source")

### 2.3 Build the daily summary model

The model reads the Source, filters three invalid statuses, and aggregates by `order_date`. dbt-doris uses `config()` to create a Doris Table with a Range Partition, Duplicate Key, and Hash Bucket.

In [ ]:
runner.show_file("Daily summary model", daily_demo_dir / "models/marts/daily_order_summary.sql")
runner.run_dbt("Create daily_order_summary", daily_demo_dir, "run", "--select", "daily_order_summary")
runner.query("Output: valid orders summarized over 3 days", """
select order_date, order_count, total_revenue
from dbt_demo_daily.daily_order_summary
order by order_date
""")

### 2.4 Run the Data Test

This step checks the daily result: dates must be non-null and unique, and order counts and revenue must be non-null.

In [ ]:
runner.show_file("Data Test definition", daily_demo_dir / "models/marts/daily_order_summary.yml")
runner.run_dbt("Test daily_order_summary", daily_demo_dir, "test", "--select", "daily_order_summary")

### 2.5 Build a monthly asynchronous materialized view

The second model reads the three rows from the previous step with `ref('daily_order_summary')`, then aggregates by month.

In [ ]:
runner.show_file("Monthly materialized view model", daily_demo_dir / "models/marts/monthly_order_summary_mv.sql")
runner.run_dbt("Create the monthly asynchronous materialized view", daily_demo_dir, "run", "--select", "monthly_order_summary_mv")
runner.run_dbt("Submit a monthly materialized view refresh", daily_demo_dir, "run", "--select", "monthly_order_summary_mv")
runner.query("Doris materialized view task", """
select MvName, Status, CreateTime
from tasks('type'='mv')
where MvDatabaseName = 'dbt_demo_daily'
  and MvName = 'monthly_order_summary_mv'
order by CreateTime desc
limit 1
""")

### 2.6 Verify the full data path and inspect the final result

The verifier waits for the asynchronous refresh, then checks the daily data, the Table Key/Partition/Bucket DDL, and the monthly result.

In [ ]:
runner.run_script("Verify the complete daily order data path", daily_demo_dir / "scripts/verify.sh")
runner.query("Final result: monthly order summary", """
select order_month, order_count, total_revenue
from dbt_demo_daily.monthly_order_summary_mv
order by order_month
""")

## Complete

The daily summary table, four Data Tests, Doris table definition, and monthly asynchronous materialized view all passed verification.